## 1D CNN (ResNet18)


### 1. 라이브러리 호출 및 seed 고정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import warnings
import torch
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from torch.utils.data import DataLoader, TensorDataset
from torch import nn, optim
from tqdm import tqdm
warnings.filterwarnings('ignore')
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

In [ ]:
# 시드 고정 코드
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### 2. 데이터 가공 및 하이퍼 파리미터 설정

In [ ]:
BATCH_SIZE = 64 # 모델이 한 번에 공부하는 데이터 묶음의 크기 (너무 많으면 메모리가 부족해지고 너무 적으면 학습이 느리고 불안정해짐)
LR = 0.001 # 모델이 배울 때 한 번에 이동하는 속도 (너무 크면 loss의 최소점을 지나치게 되고 너무 작으면 학습 속도가 느리고 local minimum에 빠짐)
EPOCH = 250 # 전체 데이터를 몇 번 반복해서 공부할지 정하는 횟수
criterion = nn.CrossEntropyLoss() # 모델의 예측이 얼마나 틀렸는지 계산하는 함수
new_model_train = True
model_type = "resnet18"
dataset = "Security"
save_model_path = f"/content/drive/MyDrive/Colab Notebooks/result/{model_type}_{dataset}.pt"
save_history_path = f"/content/drive/MyDrive/Colab Notebooks/result/{model_type}_history_{dataset}.pt"

In [ ]:
# Train, Val, Test 데이터 불러오기
train_val_DS = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train_val_data.csv', encoding='utf-8')
test_DS = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/test_data.csv', encoding='utf-8')

# Train, Val 분류를 1번만 등장한 희귀 target 제거
attack_type_counts = train_val_DS['Attack Type'].value_counts()
valid_classes = attack_type_counts[attack_type_counts >= 2].index
train_val_DS = train_val_DS[train_val_DS['Attack Type'].isin(valid_classes)]

# feature와 target 구분
X = train_val_DS['Scenario Description']
y = train_val_DS['Attack Type']

# target 라벨링
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_class = len(label_encoder.classes_)
print(num_class)

# Test Dataset을 Train, Val Dataset에 맞춰서 라벨링 일관성 유
test_DS = test_DS[test_DS["Attack Type"].isin(label_encoder.classes_)]
test_X = test_DS["Scenario Description"]
test_y = test_DS["Attack Type"]
test_y_encoded = label_encoder.transform(test_y)

# Train, Val 데이터 구분
X_train_DS, X_val_DS, y_train_DS, y_val_DS = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


# TF-IDF 벡터 변환 파이프라인 (영어 불용어 제거 포함)
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english'))
])

# Train, Val, Test feature를 TF-IDF 벡터로 변환
X_train_tfidf = pipeline.fit_transform(X_train_DS)
X_val_tfidf = pipeline.transform(X_val_DS)
X_test_tfidf = pipeline.transform(test_X)

# tensor 형태로 전
X_train_tensor = torch.tensor(X_train_tfidf.toarray(), dtype=torch.float32).unsqueeze(1) # unsqueeze(1) → (batch, 1, feature_dim) 형태로 변경
X_val_tensor = torch.tensor(X_val_tfidf.toarray(), dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_tfidf.toarray(), dtype=torch.float32).unsqueeze(1)

# 라벨도 텐서로 변환
y_train_tensor = torch.tensor(y_train_DS, dtype=torch.long)
y_val_tensor = torch.tensor(y_val_DS, dtype=torch.long)
y_test_tensor = torch.tensor(test_y_encoded, dtype=torch.long)

# tensor 형태 데이터셋 생성 (feature + target 묶음)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# 데이터 로더 생성
train_DL = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_DL   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_DL = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(X_train_tensor.shape)
print(X_val_tensor.shape)
print(X_test_tensor.shape)

### 3. 학습 및 테스트 함수 모음

In [ ]:
def Train(model, train_DL, val_DL, criterion, optimizer,
          EPOCH, BATCH_SIZE,
          save_model_path, save_history_path):

    loss_history = {"train" : [], "val":[]}
    acc_history = {"train" : [], "val":[]}
    best_loss = 9999
    for ep in range(EPOCH):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"Epoch : {ep+1}, current_LR = {current_lr}")

        model.train() # train mode로 전환
        train_loss, train_acc, _ = loss_epoch(model, train_DL, criterion, optimizer)
        loss_history["train"] += [train_loss]
        acc_history["train"] += [train_acc]

        model.eval() # test mode로 전환
        with torch.no_grad():
            val_loss, val_acc, _ = loss_epoch(model, val_DL, criterion)
            loss_history["val"] += [val_loss]
            acc_history["val"] += [val_acc]

            if val_loss < best_loss: # early stopping
                best_loss = val_loss
                torch.save({
                            "model_state_dict": model.state_dict(),
                            "optimizer_state_dict": optimizer.state_dict(),
                            "epoch": ep}, save_model_path)

        # print loss
        print(f"train loss: {round(train_loss,5)}, "
              f"val loss: {round(val_loss,5)} \n"
              f"train acc: {round(train_acc,1)} %, "
              f"val acc: {round(val_acc,1)} %, time: {round(time.time()-epoch_start)} s")
        print("-"*20)

    torch.save({"loss_history" : loss_history,
                "acc_history" : acc_history,
                "EPOCH": EPOCH,
                "BATCH_SIZE": BATCH_SIZE}, save_history_path)

    return loss_history

def Test(model, test_DL, criterion):
    # test mode로 전환
    model.eval()
    with torch.no_grad():
        test_loss, test_acc, rcorrect = loss_epoch(model, test_DL, criterion)
    print()
    print(f"Test loss : {round(test_loss,5)}")
    print(f"Test accuracy : {rcorrect}/{len(test_DL.dataset)} ({round(test_acc,1)} %)")
    return round(test_acc,1)

def loss_epoch(model, DL, criterion, optimizer = None):
    N = len(DL.dataset) # the number of data
    rloss = 0; rcorrect = 0
    for x_batch, y_batch in tqdm(DL, leave=False):
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        # inference
        y_hat = model(x_batch)
        # loss
        loss = criterion(y_hat, y_batch)
        # update
        if optimizer is not None:
            optimizer.zero_grad() # gradient 누적을 막기 위한 초기화
            loss.backward() # backpropagation
            optimizer.step() # weight update
        # loss accumulation
        loss_b = loss.item() * x_batch.shape[0]
        rloss += loss_b # running loss
        # accuracy accumulation
        pred = torch.argmax(y_hat, dim=1)
        corrects_b = torch.sum(pred == y_batch).item()
        rcorrect += corrects_b
    loss_e = rloss/N
    accruracy_e = rcorrect/N * 100

    return loss_e, accruracy_e, rcorrect

def Show_random_prediction(model, test_dataset, test_X, label_encoder):
    model.eval()

    # test_X 인덱스 리셋 (원본은 변경하지 않음)
    test_X_reset = test_X.reset_index(drop=True)

    # 랜덤 샘플 1개 선택
    random_idx = np.random.randint(0, len(test_dataset))
    x_sample, y_true = test_dataset[random_idx]

    # 예측
    with torch.no_grad():
        x_sample_input = x_sample.unsqueeze(0).to(DEVICE)
        y_hat = model(x_sample_input)
        y_pred = torch.argmax(y_hat, dim=1).item()

    # 인코딩된 레이블을 원래 레이블로 변환
    true_label_name = label_encoder.inverse_transform([y_true])[0]
    pred_label_name = label_encoder.inverse_transform([y_pred])[0]

    # 원본 텍스트 가져오기 (리셋된 인덱스 사용)
    original_text = test_X_reset.iloc[random_idx]

    # 결과 출력
    is_correct = (y_pred == y_true)

    print(f'{"CORRECT" if is_correct else "INCORRECT"}')
    print('-' * 60)
    print(f'Text: {original_text}')
    print(f'True: {true_label_name}')
    print(f'Predicted: {pred_label_name}')

def predict_single_text(model, pipeline, label_encoder, text, device="cuda"):
    # 1. TF-IDF 변환
    X_tfidf = pipeline.transform([text])
    X_tensor = torch.tensor(X_tfidf.toarray(), dtype=torch.float32).unsqueeze(1).to(device)

    # 2. 모델 예측
    model.eval()
    with torch.no_grad():
        y_hat = model(X_tensor)
        pred_idx = torch.argmax(y_hat, dim=1).item()

    # 3. 정수 → Attack Type 라벨로 역변환
    pred_label = label_encoder.inverse_transform([pred_idx])[0]

    return pred_label

### 4. Resnet 모델 구현

In [ ]:
# 기존 Resnet 모델을 1D 형태로 변환하여 사용 (새로운 model을 만드는데에 어렵고 이미 검증된 모델을 사용하기 위함)
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_channels, inner_channels, stride=1, projection=None):
        super().__init__()

        self.residual = nn.Sequential(
            nn.Conv1d(in_channels, inner_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm1d(inner_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(inner_channels, inner_channels * self.expansion, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(inner_channels * self.expansion)
        )

        self.projection = projection
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = self.residual(x)

        shortcut = self.projection(x) if self.projection is not None else x

        return self.relu(residual + shortcut)

class ResNet(nn.Module):
    def __init__(self, block, num_block_list, num_classes=916, zero_init_residual=True, input_channels=1):
        super().__init__()

        self.in_channels = 64

        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        self.stage1 = self.make_stage(block, 64, num_block_list[0], stride=1)
        self.stage2 = self.make_stage(block, 128, num_block_list[1], stride=2)
        self.stage3 = self.make_stage(block, 256, num_block_list[2], stride=2)
        self.stage4 = self.make_stage(block, 512, num_block_list[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool1d(1)

        # overfitting을 방지하기 위해 Dropout과 ReLU함수 사용
        self.fc = nn.Sequential(
            nn.Linear(512 * block.expansion, 1024),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(1024, 2048),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(1024, num_classes)
        )

        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, block):
                    nn.init.constant_(m.residual[-1].weight, 0)

    def make_stage(self, block, inner_channels, num_blocks, stride=1):

        projection = None
        if stride != 1 or self.in_channels != inner_channels * block.expansion:
            projection = nn.Sequential(
                nn.Conv1d(self.in_channels, inner_channels * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(inner_channels * block.expansion)
            )

        layers = [block(self.in_channels, inner_channels, stride, projection)]
        self.in_channels = inner_channels * block.expansion

        for _ in range(1, num_blocks):
            layers.append(block(self.in_channels, inner_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

In [ ]:
# 데이터의 class 개수 및 데이터의 양에 비해서 다른 모델들은 거대하다고 생각해서 ResNet18 사용
def resnet18(**kwargs):
    return ResNet(BasicBlock, [2, 2, 2, 2], **kwargs)

In [ ]:
# 모델 구조 확인
exec(f"model = {model_type}().to(DEVICE)")
print(model)
x_batch, _ = next(iter(train_DL))
print(model(x_batch.to(DEVICE)).shape)

### 5. 모델 학습

In [ ]:
if new_model_train:
    optimizer = optim.Adam(model.parameters(), lr = LR)
    loss_history = Train(model, train_DL, val_DL, criterion, optimizer, EPOCH,
                         BATCH_SIZE, save_model_path, save_history_path)
    loss_history = list(loss_history.values())[0]
    plt.plot(range(1, EPOCH+1), loss_history)
    plt.xlabel('Epoch')
    plt.ylabel('loss')
    plt.title("Train Loss")
    plt.grid()
else:
    optimizer = optim.Adam(load_model.parameters(), lr = LR)
    loss_history = Train(load_model, train_DL, val_DL, criterion, optimizer, EPOCH,
                         BATCH_SIZE, save_model_path, save_history_path)
    loss_history = list(loss_history.values())[0]
    plt.plot(range(1, EPOCH+1), loss_history)
    plt.xlabel('Epoch')
    plt.ylabel('loss')
    plt.title("Train Loss")
    plt.grid()

### 6. 모델 테스트

In [ ]:
# model 불러오기
check = torch.load(save_model_path)
load_model = model
load_model.load_state_dict(check['model_state_dict'])

# test 실행
Test(load_model, test_DL, criterion)

In [ ]:
# test 데이터셋에서 랜덤으로 한 개의 샘플을 뽑아 확인
Show_random_prediction(load_model, test_dataset, test_X, label_encoder)

In [ ]:
# 실제 서비스에서 사용할 때의 느낌
sample_text = "HTTP request flood detected from abnormal IP range"

pred = predict_single_text(load_model, pipeline, label_encoder, sample_text, DEVICE)
print("Predicted Attack Type:", pred)